# Task 1 - Data Preprocessing and Exploratory Data Analysis
## Portfolio: TSLA, BND, SPY | Period: 2015-2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
print("Libraries loaded")

## 1. Load and Inspect Data

In [ ]:
df = pd.read_csv("../data/processed/prices.csv", index_col=0, parse_dates=True)
print("Shape:", df.shape)
print("
Date range:", df.index[0], "to", df.index[-1])
print("
Missing values:
", df.isnull().sum())
print("
Basic stats:
")
df.describe().round(2)

## 2. Closing Price Trends

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
colors = ["#e74c3c", "#3498db", "#2ecc71"]
for i, (col, color) in enumerate(zip(df.columns, colors)):
    axes[i].plot(df.index, df[col], color=color, linewidth=1)
    axes[i].set_title(f"{col} Closing Price (2015-2026)", fontsize=13, fontweight="bold")
    axes[i].set_ylabel("Price (USD)")
    axes[i].fill_between(df.index, df[col], alpha=0.1, color=color)
plt.tight_layout()
plt.savefig("../data/processed/closing_prices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 1: Closing prices saved")

## 3. Daily Returns and Volatility

In [ ]:
returns = df.pct_change().dropna()
returns.columns = [c + "_return" for c in df.columns]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
colors = ["#e74c3c", "#3498db", "#2ecc71"]
for i, (col, color) in enumerate(zip(returns.columns, colors)):
    axes[i].plot(returns.index, returns[col]*100, color=color, linewidth=0.7, alpha=0.8)
    axes[i].set_title(f"{col.replace(chr(95)+chr(114)+chr(101)+chr(116)+chr(117)+chr(114)+chr(110), chr(32)+chr(68)+chr(97)+chr(105)+chr(108)+chr(121)+chr(32)+chr(82)+chr(101)+chr(116)+chr(117)+chr(114)+chr(110))} (%)", fontsize=12)
    axes[i].axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig("../data/processed/daily_returns.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 2: Daily returns saved")
print("
Return statistics:")
returns.describe().round(4)

## 4. Rolling Volatility (30-day)

In [ ]:
rolling_vol = returns.rolling(30).std() * np.sqrt(252)
fig, ax = plt.subplots(figsize=(14, 6))
for col, color, label in zip(rolling_vol.columns, ["#e74c3c","#3498db","#2ecc71"], ["TSLA","BND","SPY"]):
    ax.plot(rolling_vol.index, rolling_vol[col]*100, label=label, color=color, linewidth=1.2)
ax.set_title("30-Day Rolling Annualized Volatility (%)", fontsize=13, fontweight="bold")
ax.set_ylabel("Volatility (%)")
ax.legend()
plt.tight_layout()
plt.savefig("../data/processed/rolling_volatility.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 3: Rolling volatility saved")

## 5. Stationarity Test (Augmented Dickey-Fuller)

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna())
    print(f"
{name}:")
    print(f"  ADF Statistic: {result[0]:.4f}")
    print(f"  p-value: {result[1]:.4f}")
    print(f"  Stationary: {result[1] < 0.05}")
    return result[1] < 0.05

print("=== Closing Prices (Raw) ===")
for col in df.columns:
    adf_test(df[col], col)

print("
=== Daily Returns ===")
for col in returns.columns:
    adf_test(returns[col], col)

## 6. Risk Metrics: VaR and Sharpe Ratio

In [ ]:
print("=== Value at Risk (95% confidence) ===")
for col in returns.columns:
    var_95 = np.percentile(returns[col], 5)
    print(f"{col}: {var_95*100:.2f}%")

print("
=== Sharpe Ratio (annualized, risk-free=2%) ===")
rf = 0.02 / 252
for col in returns.columns:
    excess = returns[col] - rf
    sharpe = (excess.mean() / excess.std()) * np.sqrt(252)
    print(f"{col}: {sharpe:.2f}")

print("
=== Correlation Matrix ===")
corr = returns.corr()
print(corr.round(3))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", ax=ax)
ax.set_title("Asset Return Correlation Matrix")
plt.tight_layout()
plt.savefig("../data/processed/correlation.png", dpi=150, bbox_inches="tight")
plt.show()